In [1]:
import os
os.chdir(r"C:\Users\Lenovo\Desktop\KYC_Project")
print("Working directory:", os.getcwd())

Working directory: C:\Users\Lenovo\Desktop\KYC_Project


In [2]:
import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO
from skimage.metrics import structural_similarity as ssim

print("All imports successful")

All imports successful


In [3]:
doc_model = YOLO("ml/docservice/models/doc_field_detector_v3.pt")
print("YOLO model loaded")
print("Classes:", doc_model.names)

YOLO model loaded
Classes: {0: 'c_no', 1: 'emblem', 2: 'fname', 3: 'gender', 4: 'logo', 5: 'mname', 6: 'name', 7: 'photo'}


In [4]:
def detect_and_crop(image_path, conf_threshold=0.5):
    results = doc_model.predict(source=image_path, conf=conf_threshold, verbose=False)
    original_image = Image.open(image_path).convert("RGB")
    detections = {}
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        cls_name = doc_model.names[cls_id]
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        if cls_name in detections and detections[cls_name]["confidence"] >= conf:
            continue
        crop = original_image.crop((x1, y1, x2, y2))
        detections[cls_name] = {"crop": crop, "confidence": conf, "bbox": (x1, y1, x2, y2)}
    return detections

# Find a good reference image — pick one with high confidence
os.makedirs("ml/stamp-service/references", exist_ok=True)

valid_images = os.listdir("valid/images")
best_emblem = {"conf": 0, "img": None}
best_logo   = {"conf": 0, "img": None}

for fname in valid_images:
    img_path = os.path.join("valid", "images", fname)
    detections = detect_and_crop(img_path)

    if "emblem" in detections:
        if detections["emblem"]["confidence"] > best_emblem["conf"]:
            best_emblem["conf"] = detections["emblem"]["confidence"]
            best_emblem["img"]  = detections["emblem"]["crop"]

    if "logo" in detections:
        if detections["logo"]["confidence"] > best_logo["conf"]:
            best_logo["conf"] = detections["logo"]["confidence"]
            best_logo["img"]  = detections["logo"]["crop"]

# Save best reference stamps
if best_emblem["img"]:
    best_emblem["img"].save("ml/stamp-service/references/reference_emblem.jpg")
    print(f"Emblem saved — confidence: {best_emblem['conf']:.2f}, size: {best_emblem['img'].size}")

if best_logo["img"]:
    best_logo["img"].save("ml/stamp-service/references/reference_logo.jpg")
    print(f"Logo saved   — confidence: {best_logo['conf']:.2f}, size: {best_logo['img'].size}")

Emblem saved — confidence: 0.93, size: (124, 112)
Logo saved   — confidence: 0.98, size: (162, 126)


In [5]:
def orb_similarity(img1, img2):
    """Compare two images using ORB keypoint matching."""
    # Convert to grayscale numpy arrays
    if isinstance(img1, Image.Image):
        img1 = np.array(img1.convert("L"))
    if isinstance(img2, Image.Image):
        img2 = np.array(img2.convert("L"))

    # Resize to same size
    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    # ORB detector
    orb = cv2.ORB_create(nfeatures=500)
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)

    if des1 is None or des2 is None:
        return 0.0

    # Match keypoints
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)

    if len(matches) == 0:
        return 0.0

    # Score = good matches / total keypoints
    good_matches = [m for m in matches if m.distance < 50]
    score = len(good_matches) / max(len(kp1), len(kp2))
    return min(score, 1.0)


def ssim_similarity(img1, img2):
    """Compare two images using structural similarity."""
    if isinstance(img1, Image.Image):
        img1 = np.array(img1.convert("L"))
    if isinstance(img2, Image.Image):
        img2 = np.array(img2.convert("L"))

    # Resize to same size
    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    score, _ = ssim(img1, img2, full=True)
    return max(score, 0.0)  # clamp negative values to 0


def compare_stamps(crop, reference_path, threshold=0.4):
    """
    Compare detected stamp crop with reference stamp.
    Uses both ORB and SSIM, averages the scores.
    """
    reference = Image.open(reference_path).convert("RGB")

    orb_score  = orb_similarity(crop, reference)
    ssim_score = ssim_similarity(crop, reference)
    final_score = (orb_score + ssim_score) / 2

    return {
        "is_genuine": final_score >= threshold,
        "score": round(final_score, 4),
        "orb_score": round(orb_score, 4),
        "ssim_score": round(ssim_score, 4),
    }

print("Stamp comparison functions ready")

Stamp comparison functions ready


In [6]:
# Test: compare reference with itself — should be near 1.0
ref_emblem = Image.open("ml/stamp-service/references/reference_emblem.jpg")
ref_logo   = Image.open("ml/stamp-service/references/reference_logo.jpg")

result_emblem = compare_stamps(ref_emblem, "ml/stamp-service/references/reference_emblem.jpg")
result_logo   = compare_stamps(ref_logo,   "ml/stamp-service/references/reference_logo.jpg")

print("Emblem vs itself:")
print(f"  Score: {result_emblem['score']}  ORB: {result_emblem['orb_score']}  SSIM: {result_emblem['ssim_score']}")
print(f"  Genuine: {result_emblem['is_genuine']}")
print()
print("Logo vs itself:")
print(f"  Score: {result_logo['score']}  ORB: {result_logo['orb_score']}  SSIM: {result_logo['ssim_score']}")
print(f"  Genuine: {result_logo['is_genuine']}")

Emblem vs itself:
  Score: 1.0  ORB: 1.0  SSIM: 1.0
  Genuine: True

Logo vs itself:
  Score: 1.0  ORB: 1.0  SSIM: 1.0
  Genuine: True


In [7]:
def verify_stamp(citizenship_image_path, threshold=0.4):
    """
    Full pipeline:
    1. YOLO detects emblem and logo fields
    2. Compare each with reference
    3. Return combined result
    """
    detections = detect_and_crop(citizenship_image_path)

    results = {}

    for stamp_type in ["emblem", "logo"]:
        ref_path = f"ml/stamp-service/references/reference_{stamp_type}.jpg"

        if stamp_type not in detections:
            results[stamp_type] = {
                "is_genuine": False,
                "score": 0.0,
                "reason": "not detected by YOLO"
            }
            continue

        crop = detections[stamp_type]["crop"]
        comparison = compare_stamps(crop, ref_path, threshold)
        comparison["reason"] = "ok"
        comparison["yolo_confidence"] = round(detections[stamp_type]["confidence"], 3)
        results[stamp_type] = comparison

    # Overall result — both must pass
    both_genuine = all(r["is_genuine"] for r in results.values())
    avg_score = round(sum(r["score"] for r in results.values()) / len(results), 4)

    return {
        "is_genuine": both_genuine,
        "avg_score": avg_score,
        "emblem": results.get("emblem"),
        "logo": results.get("logo")
    }

print("verify_stamp pipeline ready")

verify_stamp pipeline ready


In [8]:
valid_images = os.listdir("valid/images")

for fname in valid_images[:5]:
    img_path = os.path.join("valid", "images", fname)
    result = verify_stamp(img_path)

    print(f"{fname[:40]}")
    print(f"  Overall genuine: {result['is_genuine']}  avg_score: {result['avg_score']}")
    print(f"  Emblem — score: {result['emblem']['score']}  genuine: {result['emblem']['is_genuine']}")
    print(f"  Logo   — score: {result['logo']['score']}  genuine: {result['logo']['is_genuine']}")
    print()

Data Set 14_front-nagarikta_aug1_noisy_j
  Overall genuine: False  avg_score: 0.3735
  Emblem — score: 0.4556  genuine: True
  Logo   — score: 0.2914  genuine: False

Data Set 14_front-nagarikta_aug2_phone_t
  Overall genuine: True  avg_score: 0.597
  Emblem — score: 0.6807  genuine: True
  Logo   — score: 0.5132  genuine: True

Data Set 14_front-nagarikta_aug3_blur_jp
  Overall genuine: True  avg_score: 0.7066
  Emblem — score: 0.7972  genuine: True
  Logo   — score: 0.616  genuine: True

Data Set 14_front-nagarikta_jpg.rf.leBP6
  Overall genuine: True  avg_score: 0.6058
  Emblem — score: 0.6315  genuine: True
  Logo   — score: 0.5802  genuine: True

Data Set 15_front-nagarikta_aug1_noisy_j
  Overall genuine: False  avg_score: 0.3686
  Emblem — score: 0.4625  genuine: True
  Logo   — score: 0.2747  genuine: False



In [9]:
import shutil

os.makedirs("ml/stamp-service/references", exist_ok=True)

summary = """
Stamp Verification Module
--------------------------
Methods: ORB keypoint matching + SSIM structural similarity
Classes: emblem (class 1), logo (class 4)
Threshold: 0.4 (configurable)

References:
  ml/stamp-service/references/reference_emblem.jpg
  ml/stamp-service/references/reference_logo.jpg

Functions:
  orb_similarity(img1, img2)         -> float
  ssim_similarity(img1, img2)        -> float
  compare_stamps(crop, ref_path)     -> dict
  verify_stamp(citizenship_path)     -> dict
"""

with open("ml/stamp-service/stamp_summary.txt", "w") as f:
    f.write(summary)

print("Module saved")
print(summary)

Module saved

Stamp Verification Module
--------------------------
Methods: ORB keypoint matching + SSIM structural similarity
Classes: emblem (class 1), logo (class 4)
Threshold: 0.4 (configurable)

References:
  ml/stamp-service/references/reference_emblem.jpg
  ml/stamp-service/references/reference_logo.jpg

Functions:
  orb_similarity(img1, img2)         -> float
  ssim_similarity(img1, img2)        -> float
  compare_stamps(crop, ref_path)     -> dict
  verify_stamp(citizenship_path)     -> dict

